In [2]:
# Install required libraries (uncomment the line below if running in a fresh Colab environment)
# !pip install transformers torch

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# ==========================================
# 1. Model Loading (Mandatory)
# ==========================================
# We are using DialoGPT as it is specifically fine-tuned for conversational responses.
model_name = "microsoft/DialoGPT-medium"

print(f"Downloading/Loading the model '{model_name}'... This may take a minute.")
# Load the tokenizer and the pre-trained transformer model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)
print("Model loaded successfully!\n")
print("-" * 50)

def chat_with_bot():
    """
    Main function to handle the conversation loop.
    """
    # Initialize chat history to keep track of the conversation flow
    chat_history_ids = None

    print("Chatbot: Hello! I am your AI assistant. How can I help you today?")

    # ==========================================
# 4. Continuous Conversation Loop
    # ==========================================
    step = 0
    while True:
        # 2. User Input Handling
        user_input = input("User: ")

        # 5. Exit Condition
        # Stop chatbot when user types exit or quit
        if user_input.lower() in ['exit', 'quit']:
            print("Chatbot: Goodbye! Have a great day.")
            break

        # Encode the new user input, add the eos_token (end of string) and return a tensor in Pytorch
        new_user_input_ids = tokenizer.encode(user_input + tokenizer.eos_token, return_tensors='pt')

        # Append the new user input tokens to the chat history (if history exists)
        # This is crucial for maintaining conversation context
        if step > 0:
            bot_input_ids = torch.cat([chat_history_ids, new_user_input_ids], dim=-1)
        else:
            bot_input_ids = new_user_input_ids

        # ==========================================
        # 3. Response Generation
        # ==========================================
        # Generate a response while limiting the total chat history to 1000 tokens
        # pad_token_id is set to eos_token_id to avoid warnings
        chat_history_ids = model.generate(
            bot_input_ids,
            max_length=1000,
            pad_token_id=tokenizer.eos_token_id,
            no_repeat_ngram_size=3,       # Prevents the model from repeating the same phrases
            do_sample=True,               # Enables sampling to make responses more dynamic
            top_k=100,
            top_p=0.7,
            temperature=0.8               # Controls randomness/creativity of the response
        )

        # Decode the generated response, skipping the previous chat history tokens
        # We slice [bot_input_ids.shape[-1]:] to only get the newest generated tokens
        response = tokenizer.decode(chat_history_ids[:, bot_input_ids.shape[-1]:][0], skip_special_tokens=True)

        # Display Output
        print(f"Chatbot: {response}")

        step += 1

# Start the chatbot
if __name__ == "__main__":
    chat_with_bot()

Downloading/Loading the model 'microsoft/DialoGPT-medium'... This may take a minute.


Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: microsoft/DialoGPT-medium
Key                              | Status     |  | 
---------------------------------+------------+--+-
transformer.h.{0...23}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully!

--------------------------------------------------
Chatbot: Hello! I am your AI assistant. How can I help you today?
User: EXIT
Chatbot: Goodbye! Have a great day.
